In [9]:
import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
from qiskit.circuit import QuantumCircuit, ParameterVector
import math

CURRENT_SEED = 17
COMPONENTS   = [32, 16, 8, 4]
N_QUBITS = 6

os.makedirs('../compressedFeatures', exist_ok=True)

# Angular Scaling
Scala le feature compresse in (0, 2π] con min-max. Fit solo su train e transform su tutti e tre gli split.

In [3]:
def angular_scaling(X_train, X_val, X_test):

    scaler = MinMaxScaler(feature_range=(1e-6, 2 * np.pi)) #1e-6 perchè nn deve essere 0
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [ ]:
for d in COMPONENTS:

    path = f'../compressedFeatures/pca_d{d}_seed_{CURRENT_SEED}.pt'
    ckpt = torch.load(path, weights_only=False)

    X_train = ckpt['train'].numpy()
    X_val   = ckpt['val'].numpy()
    X_test  = ckpt['test'].numpy()


    X_train_ang, X_val_ang, X_test_ang, scaler = angular_scaling(
        X_train, X_val, X_test
    )

    print(f'd={d:2d} \nrange train: [{X_train_ang.min():.4f}, {X_train_ang.max():.4f}]')
    print(f'range val:   [{X_val_ang.min():.4f}, {X_val_ang.max():.4f}]')
    print(f'range test:  [{X_test_ang.min():.4f}, {X_test_ang.max():.4f}]\n')


    torch.save({
        'train_ang':       torch.from_numpy(X_train_ang).float(),
        'val_ang':         torch.from_numpy(X_val_ang).float(),
        'test_ang':        torch.from_numpy(X_test_ang).float(),
        'y_train':         ckpt['y_train'],
        'y_val':           ckpt['y_val'],
        'y_test':          ckpt['y_test'],
        'd':               d,
        'seed':            CURRENT_SEED,
        'angular_scaling': 'minmax_(0,2pi]'
    }, f'../compressedFeatures/angular_d{d}_seed_{CURRENT_SEED}.pt')

d=32 
range train: [0.0000, 6.2832]
range val:   [-2.1661, 11.7790]
range test:  [-0.4066, 7.5066]

d=16 
range train: [0.0000, 6.2832]
range val:   [-2.1661, 11.7790]
range test:  [-0.4066, 6.9193]

d= 8 
range train: [0.0000, 6.2832]
range val:   [-0.3023, 11.7790]
range test:  [-0.2624, 5.9727]

d= 4 
range train: [0.0000, 6.2832]
range val:   [-0.1169, 11.7790]
range test:  [0.0394, 5.0199]



# Angle Encoding con Data Re-Uploading

In [31]:
def build_encoding_layer(d, n_qubits, block_idx):

    # Costruisce un layer di encoding RY per un singolo blocco. Restituisce il circuito e i parametri di encoding del blocco.

    params = ParameterVector(f'x_{block_idx}', n_qubits) # ParameterVector è una lista di variabili simboliche non ha ancora valori numerici, sono solo nomi che venmgono sostituiti quando si esegue il circuito
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(params[i], i)
    return qc, params

def build_encoding_circuit(d, n_qubits=N_QUBITS):

    # Costruisce il circuito di encoding completo con data re-uploading. Per ogni blocco di nqubits feature applica RY su tutti i qubit.
    # Se d non è divisibile per n_qubits, l'ultimo blocco viene zero-paddato.

    n_blocks = math.ceil(d / n_qubits)
    
    qc = QuantumCircuit(n_qubits)
    all_input_params = []

    for block in range(n_blocks):
        enc_layer, enc_params = build_encoding_layer(d, n_qubits, block)
        qc.compose(enc_layer, inplace=True)
        all_input_params.extend(enc_params)

    return qc, all_input_params, n_blocks

# verifica per ogni d
for d in COMPONENTS:
    qc, input_params, n_blocks = build_encoding_circuit(d, N_QUBITS)
    print(f'd={d:2d} | blocchi={n_blocks} | parametri encoding={len(input_params)} | padding={n_blocks*N_QUBITS - d}')


d=32 | blocchi=6 | parametri encoding=36 | padding=4
d=16 | blocchi=3 | parametri encoding=18 | padding=2
d= 8 | blocchi=2 | parametri encoding=12 | padding=4
d= 4 | blocchi=1 | parametri encoding=6 | padding=2


In [26]:
qc_example, _, _ = build_encoding_circuit(d=32, n_qubits=N_QUBITS)
qc_example.draw('text')
#qc_example.draw('mpl')
#qc_example.draw('latex')

┌────────────┐┌────────────┐┌────────────┐┌────────────┐┌────────────┐»
q_0: ┤ Ry(x_0[0]) ├┤ Ry(x_1[0]) ├┤ Ry(x_2[0]) ├┤ Ry(x_3[0]) ├┤ Ry(x_4[0]) ├»
     ├────────────┤├────────────┤├────────────┤├────────────┤├────────────┤»
q_1: ┤ Ry(x_0[1]) ├┤ Ry(x_1[1]) ├┤ Ry(x_2[1]) ├┤ Ry(x_3[1]) ├┤ Ry(x_4[1]) ├»
     ├────────────┤├────────────┤├────────────┤├────────────┤├────────────┤»
q_2: ┤ Ry(x_0[2]) ├┤ Ry(x_1[2]) ├┤ Ry(x_2[2]) ├┤ Ry(x_3[2]) ├┤ Ry(x_4[2]) ├»
     ├────────────┤├────────────┤├────────────┤├────────────┤├────────────┤»
q_3: ┤ Ry(x_0[3]) ├┤ Ry(x_1[3]) ├┤ Ry(x_2[3]) ├┤ Ry(x_3[3]) ├┤ Ry(x_4[3]) ├»
     ├────────────┤├────────────┤├────────────┤├────────────┤├────────────┤»
q_4: ┤ Ry(x_0[4]) ├┤ Ry(x_1[4]) ├┤ Ry(x_2[4]) ├┤ Ry(x_3[4]) ├┤ Ry(x_4[4]) ├»
     ├────────────┤├────────────┤├────────────┤├────────────┤├────────────┤»
q_5: ┤ Ry(x_0[5]) ├┤ Ry(x_1[5]) ├┤ Ry(x_2[5]) ├┤ Ry(x_3[5]) ├┤ Ry(x_4[5]) ├»
     └────────────┘└────────────┘└────────────┘└────────────┘└────────────┘»
«     ┌────────────┐
«q_0: ┤ Ry(x_5[0]) ├
«     ├────────────┤
«q_1: ┤ Ry(x_5[1]) ├
«     ├────────────┤
«q_2: ┤ Ry(x_5[2]) ├
«     ├────────────┤
«q_3: ┤ Ry(x_5[3]) ├
«     ├────────────┤
«q_4: ┤ Ry(x_5[4]) ├
«     ├────────────┤
«q_5: ┤ Ry(x_5[5]) ├
«     └────────────┘